# COAT — Stage-1 baselines (Random / BCE / InfoNCE / AdvInfoNCE / MyLoss) — Hit@20 & NDCG@20, all-ranking, early-stopped

Train split = `train.csv` (MNAR, user-selected ratings) — used **100% intact** to fit every model, unaffected by the validation split below.
`random.csv` (MCAR, randomly-exposed ratings) is split **unevenly** into a validation slice and a test slice.

Changes vs. `coat-all-baselines.ipynb`:

1. **CE renamed to BCE.** The loss was always `F.binary_cross_entropy_with_logits` (sigmoid, point-wise over the in-batch B×B pair matrix with `pos_weight=B-1`), never a softmax/categorical cross-entropy. Only the name was wrong; the implementation is unchanged.
2. **Fixed-epoch training replaced with early stopping**, exactly mirroring `yahoo-earlystopping.ipynb`. Every trainable baseline is evaluated on `val_ground_truth` (all-ranking HR@20) after every epoch, up to `MAX_EPOCHS`, and stops after `PATIENCE` epochs without improvement. The encoder weights from the best validation epoch are restored before final embeddings are computed and scored on `test_ground_truth`. `Random` has no training loop, so it's unaffected.
3. **Validation carved out of `random.csv` (MCAR), unevenly — option 2, same as the Yahoo notebook.** Rather than an even 50/50 split, `random.csv` is split `VAL_FRACTION=0.2` / `1-VAL_FRACTION=0.8` in test's favor: 20% for validation/early-stopping, 80% held for the final test metric. This keeps validation drawn from the same unbiased (MCAR) distribution the test metric is measured on, while keeping most of the statistical power for the number that actually gets reported. `train.csv` (train) is untouched either way; training data utilization is unaffected by this split.
4. **Split mechanics are identical to the Yahoo notebook**: a fixed `SPLIT_SEED=2026`, independent of the training-seed loop, is used to partition `random.csv` once so every baseline/seed early-stops on the same `val_df` and is finally scored on the same `test_df` — only model training differs.

Everything else (COAT-specific data loading — separate `train.csv`/`random.csv` files, pre-indexed dense ids, one-hot `user_feat_onehot.csv`/`item_feat_onehot.csv`, `rating_max`-normalized `weighted_clip_loss` — plus the shared TwoTowerEncoder backbone, in-batch negatives, symmetric losses, optimizer/schedule, multi-seed aggregation logic) is unchanged from `coat-all-baselines.ipynb`. No Stage-2 (COAT has no timestamps).

Note: `SEEDS` is kept as `[42, 43, 44, 45]` (the original COAT notebook's 4 seeds) since only the validation-split/early-stopping strategy and the CE→BCE naming fix were requested — the Yahoo notebook's reduction to 3 seeds was called out there as a separate, dataset-specific change.

In [1]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/kkkongxin/dataset-coat/coat/random.csv
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/user_feat_onehot.csv
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/build_dataset.ipynb
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/test.ascii
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/item_feat_onehot.csv
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/user_feat_label.csv
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/propensities.ascii
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/train.ascii
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/train.csv
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/item_feat_label.csv
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/user_item_features/item_features.ascii
/kaggle/input/datasets/kkkongxin/dataset-coat/coat/user_item_features/user_features.ascii


In [2]:
"""
COAT Stage-1 baselines: Random / BCE / InfoNCE / AdvInfoNCE / MyLoss (SC-InfoNCE)

Stage-1 ONLY, two metrics: Hit@20 and NDCG@20, evaluated with the all-ranking
strategy (rank every unseen item for each test user).

Changes vs. earlier versions of this notebook:
  - The loss previously named "CE" is renamed "BCE" everywhere (baseline
    list, dispatch, function name, prints, CSV output). The implementation
    itself (F.binary_cross_entropy_with_logits, point-wise over the in-batch
    B x B pair matrix, pos_weight = B - 1) is unchanged -- it was mislabeled,
    not miscoded.
  - Validation is carved out of random.csv (MCAR), unevenly, in test's favor
    (VAL_FRACTION=0.2 val / 0.8 test), rather than training for a fixed
    number of epochs. train.csv is used 100% intact to fit every model
    regardless -- this split only affects which MCAR interactions are used
    for early stopping vs. the final reported test metric.
  - Every trainable baseline is early-stopped on val_ground_truth
    (all-ranking HR@K, monitored every epoch, patience = PATIENCE, hard cap
    MAX_EPOCHS); the best-validation-epoch encoder weights are restored
    before computing the embeddings that get scored on test_ground_truth.

Adapted from the Yahoo!R3 early-stopping script; COAT-specific data handling
(separate train/random files, pre-indexed dense ids, one-hot feature CSVs,
rating_max-normalized weighted_clip_loss) is preserved from the original
`coat-all-baselines.ipynb`.
"""

import os, math, random, time, copy
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

POS_THRESHOLD = 4       # rating >= 4 is a positive interaction (paper convention)
VAL_FRACTION = 0.2      # random.csv (MCAR) split: 20% val / 80% test -- uneven,
                         # in test's favor, to preserve statistical power on the
                         # number that actually gets reported/published
SPLIT_SEED = 2026        # fixed, independent of the training-seed loop -- the
                          # val/test partition of random.csv must be identical
                          # across every seed and every baseline for fairness


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# 1. DATA / SPLIT
#    train = train.csv (MNAR, user-selected)        -> TRAIN, used 100% intact
#    random.csv (MCAR, randomly exposed) is split    -> VAL (20%) / TEST (80%)
#    Computed once, shared by every baseline and every seed.
# ═══════════════════════════════════════════════════════════════════════
COAT_DIR = "/kaggle/input/datasets/kkkongxin/dataset-coat/coat"


def load_coat():
    train_df = pd.read_csv(f"{COAT_DIR}/train.csv")
    mcar_df = pd.read_csv(f"{COAT_DIR}/random.csv")
    user_feat_df = pd.read_csv(f"{COAT_DIR}/user_feat_onehot.csv")
    item_feat_df = pd.read_csv(f"{COAT_DIR}/item_feat_onehot.csv")

    num_users = user_feat_df.shape[0]
    num_items = item_feat_df.shape[0]

    assert train_df["user_id"].max() < num_users and mcar_df["user_id"].max() < num_users, \
        "user_feat_onehot.csv has fewer rows than the max user_id seen in train/random"
    assert train_df["item_id"].max() < num_items and mcar_df["item_id"].max() < num_items, \
        "item_feat_onehot.csv has fewer rows than the max item_id seen in train/random"

    train_df = train_df.rename(columns={"user_id": "user_idx", "item_id": "item_idx"})
    mcar_df = mcar_df.rename(columns={"user_id": "user_idx", "item_id": "item_idx"})

    train_df["user_idx"] = train_df["user_idx"].astype(int)
    train_df["item_idx"] = train_df["item_idx"].astype(int)
    mcar_df["user_idx"] = mcar_df["user_idx"].astype(int)
    mcar_df["item_idx"] = mcar_df["item_idx"].astype(int)

    # Split the MCAR file (random.csv) into val / test, unevenly, in test's
    # favor. Fixed SPLIT_SEED, independent of the training-seed loop, so
    # every baseline/seed early-stops on the same val_df and is finally
    # scored on the same test_df -- only model training differs.
    rng = np.random.default_rng(SPLIT_SEED)
    shuffled_idx = rng.permutation(len(mcar_df))
    n_val = int(len(mcar_df) * VAL_FRACTION)
    val_df = mcar_df.iloc[shuffled_idx[:n_val]].reset_index(drop=True)
    test_df = mcar_df.iloc[shuffled_idx[n_val:]].reset_index(drop=True)

    user_feat_tensor = torch.tensor(user_feat_df.values, dtype=torch.float32)
    item_feat_tensor = torch.tensor(item_feat_df.values, dtype=torch.float32)

    rating_max = float(max(train_df["rating"].max(), mcar_df["rating"].max()))
    rating_min = float(min(train_df["rating"].min(), mcar_df["rating"].min()))
    print(f"COAT: num_users={num_users}  num_items={num_items}  "
          f"train_interactions={len(train_df)}  "
          f"random.csv total={len(mcar_df)} -> val={len(val_df)} / test={len(test_df)}  "
          f"rating range=[{rating_min:.0f}, {rating_max:.0f}]  "
          f"(POS_THRESHOLD={POS_THRESHOLD}, VAL_FRACTION={VAL_FRACTION})")

    return dict(train_df=train_df, val_df=val_df, test_df=test_df,
                num_users=num_users, num_items=num_items,
                user_feat_tensor=user_feat_tensor, item_feat_tensor=item_feat_tensor,
                rating_max=rating_max)


def make_ground_truth(split_df, pos_threshold=POS_THRESHOLD):
    """uid -> set of item_idx with rating >= pos_threshold. Works for either
    val_df or test_df. Users with zero qualifying positives are dropped."""
    pos_df = split_df[split_df["rating"] >= pos_threshold]
    ground_truth = defaultdict(set)
    for row in pos_df.itertuples():
        ground_truth[row.user_idx].add(row.item_idx)
    return dict(ground_truth)


def make_r_lookup(train_df, num_users, num_items):
    r_lookup = torch.zeros(num_users, num_items, dtype=torch.float32)
    uids_all = torch.tensor(train_df["user_idx"].values, dtype=torch.long)
    iids_all = torch.tensor(train_df["item_idx"].values, dtype=torch.long)
    rats_all = torch.tensor(train_df["rating"].values, dtype=torch.float32)
    r_lookup[uids_all, iids_all] = rats_all
    return r_lookup.to(DEVICE)


def make_user_seen(train_df):
    """All train interactions (any rating) get masked out at eval time --
    for BOTH val and test, since these are train.csv interactions the model
    was actually trained on."""
    user_seen = defaultdict(set)
    for row in train_df.itertuples():
        user_seen[row.user_idx].add(row.item_idx)
    return user_seen


class CLIPDataset(Dataset):
    """Positive (rating >= POS_THRESHOLD) train pairs; returns uid+iid + raw features."""
    def __init__(self, train_df, user_feat_tensor, item_feat_tensor, pos_threshold=POS_THRESHOLD):
        pos_df = train_df[train_df["rating"] >= pos_threshold].copy()
        self.user_idxs = pos_df["user_idx"].values.astype(int)
        self.item_idxs = pos_df["item_idx"].values.astype(int)
        self.user_feats = user_feat_tensor
        self.item_feats = item_feat_tensor

    def __len__(self):
        return len(self.user_idxs)

    def __getitem__(self, idx):
        uid = int(self.user_idxs[idx])
        iid = int(self.item_idxs[idx])
        return {"uid": uid, "iid": iid,
                "user_feats": self.user_feats[uid],
                "item_feats": self.item_feats[iid]}


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# 2. SHARED ARCHITECTURE — identical across BCE / InfoNCE / AdvInfoNCE / MyLoss
#    (unchanged from coat-all-baselines.ipynb; note COAT is far smaller than
#    MovieLens/Yahoo (290 users / 300 items), so this backbone has a lot more
#    capacity than data -- kept as-is for methodological parity with the
#    reference, but HIDDEN can be shrunk if overfitting is a concern.)
# ═══════════════════════════════════════════════════════════════════════
EMBED_DIM = 256
ID_EMB_DIM = 64
HIDDEN = (256,128)          # -> 3 linear layers total per tower (2 hidden + 1 out)


def make_tower(input_dim, hidden=HIDDEN, out_dim=EMBED_DIM):
    layers, in_d = [], input_dim
    for h in hidden:
        layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU()]
        in_d = h
    layers.append(nn.Linear(in_d, out_dim))
    return nn.Sequential(*layers)


class TwoTowerEncoder(nn.Module):
    """The ONE backbone every trainable baseline shares. No loss logic here."""
    def __init__(self, num_users, num_items, user_feat_dim, item_feat_dim,
                 id_emb_dim=ID_EMB_DIM, hidden=HIDDEN, out_dim=EMBED_DIM):
        super().__init__()
        self.user_id_emb = nn.Embedding(num_users, id_emb_dim)
        self.item_id_emb = nn.Embedding(num_items, id_emb_dim)
        self.user_tower = make_tower(user_feat_dim + id_emb_dim, hidden, out_dim)
        self.item_tower = make_tower(item_feat_dim + id_emb_dim, hidden, out_dim)

    def encode_user(self, user_feats, uid):
        id_e = self.user_id_emb(uid)
        return F.normalize(self.user_tower(torch.cat([user_feats, id_e], dim=-1)), dim=-1)

    def encode_item(self, item_feats, iid):
        id_e = self.item_id_emb(iid)
        return F.normalize(self.item_tower(torch.cat([item_feats, id_e], dim=-1)), dim=-1)


class HardnessModel(nn.Module):
    """AdvInfoNCE-embed adversary: separate, smaller embedding table."""
    def __init__(self, num_users, num_items, dim=ID_EMB_DIM):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def raw_scores(self, uid, iid):
        """Full in-batch score matrix g(u_a, i_b) for a batch, shape (B,B)."""
        u = self.user_emb(uid)     # (B, d)
        v = self.item_emb(iid)     # (B, d)
        return u @ v.T             # (B, B)


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# 3. LOSS FUNCTIONS
#    Only the loss function is allowed to differ across baselines; everything
#    else is shared, per the fairness rationale in the module docstring.
# ═══════════════════════════════════════════════════════════════════════
def infonce_loss(u_emb, v_emb, temperature=0.07):
    """Symmetric in-batch InfoNCE."""
    logits = (u_emb @ v_emb.T) / temperature
    targets = torch.arange(logits.size(0), device=logits.device)
    return (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2


def weighted_clip_loss(u_emb, v_emb, uid, iid, r_lookup, rating_max, temperature=0.07):
    """MyLoss / SC-InfoNCE. Negative weight = 1/W, positive weight = W."""
    B = u_emb.size(0)
    uid = uid.clamp(0, r_lookup.size(0) - 1)
    iid = iid.clamp(0, r_lookup.size(1) - 1)

    logits = (u_emb @ v_emb.T) / temperature
    W = r_lookup[uid[:, None], iid[None, :]] / rating_max

    EPS = 1e-2
    Wc = torch.clamp(W, min=EPS, max=1.0)

    # Reciprocal weighting for negatives: 1/W
    log_weights = torch.log(1.0 / Wc)

    # Positive pairs: unchanged, weight = W
    diag_mask = torch.eye(B, device=logits.device, dtype=torch.bool)
    log_weights[diag_mask] = torch.log(
        torch.clamp(W[diag_mask], min=EPS)
    )

    weighted_logits = logits + log_weights
    targets = torch.arange(B, device=logits.device)

    return (
        F.cross_entropy(weighted_logits, targets) +
        F.cross_entropy(weighted_logits.T, targets)
    ) / 2


def advinfonce_loss(u_emb, v_emb, hardness_model, uid, iid, temperature=0.07, K=64.0):
    """In-batch AdvInfoNCE (AdvInfoNCE-embed), symmetric."""
    B = u_emb.size(0)
    logits = (u_emb @ v_emb.T) / temperature           # (B, B), s(u_a, i_b)
    diag_mask = torch.eye(B, device=logits.device, dtype=torch.bool)

    raw = hardness_model.raw_scores(uid, iid)           # (B, B)
    raw_masked = raw.masked_fill(diag_mask, float("-inf"))
    log_p = F.log_softmax(raw_masked, dim=-1)           # p(j|u,i), zero mass on diagonal

    log_K = math.log(K)
    log_weighted_neg = log_K + log_p + logits
    log_weighted_neg = log_weighted_neg.masked_fill(diag_mask, float("-inf"))

    s_pos = logits[diag_mask].view(B)
    all_terms = torch.cat([s_pos.unsqueeze(1), log_weighted_neg], dim=1)
    log_denom = torch.logsumexp(all_terms, dim=1)
    loss_row = -(s_pos - log_denom)                     # user -> item direction

    logits_T = logits.T
    raw_T = raw.T
    raw_T_masked = raw_T.masked_fill(diag_mask, float("-inf"))
    log_p_T = F.log_softmax(raw_T_masked, dim=-1)
    log_weighted_neg_T = (log_K + log_p_T + logits_T).masked_fill(diag_mask, float("-inf"))
    s_pos_T = logits_T[diag_mask].view(B)
    all_terms_T = torch.cat([s_pos_T.unsqueeze(1), log_weighted_neg_T], dim=1)
    log_denom_T = torch.logsumexp(all_terms_T, dim=1)
    loss_col = -(s_pos_T - log_denom_T)

    return (loss_row.mean() + loss_col.mean()) / 2


def bce_loss(u_emb, v_emb, temperature=0.07, pos_weight_val=None):
    """Point-wise BCE baseline over the same in-batch B x B pair matrix.
    Renamed from `ce_loss` -- this was always binary cross-entropy
    (F.binary_cross_entropy_with_logits, sigmoid) over every cell of the
    B x B pair matrix, never a softmax/categorical cross-entropy like
    InfoNCE. pos_weight = B - 1 balances the single positive cell per row
    against its B - 1 negative cells (exact negative:positive ratio)."""
    B = u_emb.size(0)
    logits = (u_emb @ v_emb.T) / temperature
    targets = torch.eye(B, device=logits.device)
    pw = torch.tensor(float(B - 1) if pos_weight_val is None else pos_weight_val,
                       device=logits.device)
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw)


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# 4. EVALUATION — all-ranking, Hit@20 and NDCG@20.
#    Used for BOTH validation (early stopping) and final test scoring.
#      Hit@20  = 1{|top20 ∩ ground_truth(u)| > 0}                (binary hit)
#      NDCG@20 = DCG@20 / IDCG@20, relevance=1 for ground-truth positives
#    both averaged over users. COAT's test set gives multiple relevant items
#    per user, so this is the standard multi-relevant-item generalization.
# ═══════════════════════════════════════════════════════════════════════
HIT_K = 20


def _dcg_at_k_multi(hit_flags, k):
    dcg = 0.0
    for rank, hit in enumerate(hit_flags[:k]):
        if hit:
            dcg += 1.0 / math.log2(rank + 2)
    return dcg


def _idcg_at_k(num_pos, k):
    n = min(num_pos, k)
    return sum(1.0 / math.log2(r + 2) for r in range(n)) if n > 0 else 0.0


def evaluate_retriever_full(user_vecs, item_vecs, ground_truth, user_seen, K=HIT_K):
    """Full ranking among all unseen items (no negative sampling). Returns Hit@K and NDCG@K.
    `ground_truth` can be either val_ground_truth or test_ground_truth -- only
    train interactions (`user_seen`) are ever masked out of the candidate pool."""
    U = torch.tensor(user_vecs, dtype=torch.float32).to(DEVICE)
    V = torch.tensor(item_vecs, dtype=torch.float32).to(DEVICE)
    hits = 0.0
    ndcg = 0.0
    total = 0

    with torch.no_grad():
        for uid, pos_items in ground_truth.items():
            if len(pos_items) == 0:
                continue
            total += 1
            seen = user_seen.get(uid, set())
            scores = (U[uid:uid + 1] @ V.T).squeeze(0).clone()
            if seen:
                scores[torch.tensor(list(seen), dtype=torch.long, device=DEVICE)] = float("-inf")
            ranked = torch.argsort(scores, descending=True).cpu().tolist()

            topk = ranked[:K]
            hit_flags = [1 if item in pos_items else 0 for item in topk]
            hits += 1.0 if any(hit_flags) else 0.0
            ndcg += _dcg_at_k_multi(hit_flags, K) / max(_idcg_at_k(len(pos_items), K), 1e-8)

    return {f"Hit@{K}": hits / total, f"NDCG@{K}": ndcg / total}


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# 5. STAGE-1 TRAINING DRIVER — dispatches on loss name, early-stops on
#    validation Hit@K, restores the best-validation-epoch encoder weights.
# ═══════════════════════════════════════════════════════════════════════
BATCH_SIZE = 128
MAX_EPOCHS = 40          # hard cap; baselines rarely need all of it once early stopping is on
PATIENCE = 7             # epochs without val-Hit@K improvement before stopping
VAL_METRIC = f"Hit@{HIT_K}"  # metric monitored for early stopping / checkpoint selection
LR = 1e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 5.0
TEMPERATURE = 0.07

# AdvInfoNCE-specific
ADV_LR = 5e-5
ADV_T_ADV = 5
ADV_E_ADV = 200
ADV_K = 64.0


def get_full_embeddings(encoder, user_feat_tensor, item_feat_tensor, num_users, num_items):
    encoder.eval()
    with torch.no_grad():
        user_vecs = []
        for s in range(0, num_users, 512):
            e = min(s + 512, num_users)
            uf = user_feat_tensor[s:e].to(DEVICE)
            uid = torch.arange(s, e, device=DEVICE)
            user_vecs.append(encoder.encode_user(uf, uid).cpu().numpy())
        user_vecs = np.vstack(user_vecs)

        item_vecs = []
        for s in range(0, num_items, 512):
            e = min(s + 512, num_items)
            itf = item_feat_tensor[s:e].to(DEVICE)
            iid = torch.arange(s, e, device=DEVICE)
            item_vecs.append(encoder.encode_item(itf, iid).cpu().numpy())
        item_vecs = np.vstack(item_vecs)
    return user_vecs, item_vecs


def train_stage1(loss_name, seed, data, train_df, r_lookup, user_seen, val_ground_truth):
    """loss_name in {"Random", "BCE", "InfoNCE", "AdvInfoNCE", "MyLoss"}.

    Trainable baselines use early stopping on val_ground_truth (Hit@K), monitored
    every epoch, with patience PATIENCE and a hard cap of MAX_EPOCHS. The encoder
    state with the best validation Hit@K is restored before returning.
    """
    set_seed(seed)
    num_users, num_items = data["num_users"], data["num_items"]
    user_feat_tensor, item_feat_tensor = data["user_feat_tensor"], data["item_feat_tensor"]
    rating_max = data["rating_max"]

    if loss_name == "Random":
        rng = np.random.default_rng(seed)
        user_vecs = rng.normal(size=(num_users, EMBED_DIM)).astype(np.float32)
        item_vecs = rng.normal(size=(num_items, EMBED_DIM)).astype(np.float32)
        user_vecs /= np.linalg.norm(user_vecs, axis=1, keepdims=True)
        item_vecs /= np.linalg.norm(item_vecs, axis=1, keepdims=True)
        return user_vecs, item_vecs, None, 0

    encoder = TwoTowerEncoder(num_users, num_items,
                               user_feat_dim=user_feat_tensor.shape[1],
                               item_feat_dim=item_feat_tensor.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)

    hardness_model = None
    opt_adv = None
    if loss_name == "AdvInfoNCE":
        hardness_model = HardnessModel(num_users, num_items).to(DEVICE)
        opt_adv = torch.optim.Adam(hardness_model.parameters(), lr=ADV_LR)

    dataset = CLIPDataset(train_df, user_feat_tensor, item_feat_tensor)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True, drop_last=True)

    global_step = 0
    adv_updates_done = 0

    best_val_metric = -1.0
    best_epoch = 0
    epochs_without_improve = 0
    best_encoder_state = copy.deepcopy(encoder.state_dict())

    for epoch in range(MAX_EPOCHS):
        encoder.train()
        total_loss, steps = 0.0, 0
        t0 = time.time()
        for batch in loader:
            uf = batch["user_feats"].to(DEVICE)
            itf = batch["item_feats"].to(DEVICE)
            uid = batch["uid"].to(DEVICE)
            iid = batch["iid"].to(DEVICE)

            u = encoder.encode_user(uf, uid)
            v = encoder.encode_item(itf, iid)

            if loss_name == "InfoNCE":
                loss = infonce_loss(u, v, TEMPERATURE)
            elif loss_name == "MyLoss":
                loss = weighted_clip_loss(u, v, uid, iid, r_lookup, rating_max, TEMPERATURE)
            elif loss_name == "BCE":
                loss = bce_loss(u, v, TEMPERATURE)
            elif loss_name == "AdvInfoNCE":
                for p in hardness_model.parameters():
                    p.requires_grad_(False)
                loss = advinfonce_loss(u, v, hardness_model, uid, iid, TEMPERATURE, ADV_K)
            else:
                raise ValueError(loss_name)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(encoder.parameters(), GRAD_CLIP)
            opt.step()
            total_loss += loss.item(); steps += 1
            global_step += 1

            # ---- AdvInfoNCE adversarial half-step (Algorithm 1) ----
            if loss_name == "AdvInfoNCE" and global_step % ADV_T_ADV == 0 \
                    and adv_updates_done < ADV_E_ADV:
                for p in encoder.parameters():
                    p.requires_grad_(False)
                for p in hardness_model.parameters():
                    p.requires_grad_(True)

                u2 = encoder.encode_user(uf, uid)
                v2 = encoder.encode_item(itf, iid)
                adv_loss = advinfonce_loss(u2, v2, hardness_model, uid, iid, TEMPERATURE, ADV_K)
                opt_adv.zero_grad()
                (-adv_loss).backward()      # maximize
                opt_adv.step()
                adv_updates_done += 1

                for p in encoder.parameters():
                    p.requires_grad_(True)

        sched.step()

        # ---- validation pass: all-ranking Hit@K/NDCG@K on val_ground_truth ----
        val_user_vecs, val_item_vecs = get_full_embeddings(
            encoder, user_feat_tensor, item_feat_tensor, num_users, num_items)
        val_metrics = evaluate_retriever_full(val_user_vecs, val_item_vecs, val_ground_truth,
                                               user_seen, K=HIT_K)
        val_score = val_metrics[VAL_METRIC]

        print(f"  [{loss_name} seed={seed}] epoch {epoch+1}/{MAX_EPOCHS} "
              f"loss={total_loss/steps:.4f} val_{VAL_METRIC}={val_score*100:.2f} "
              f"time={time.time()-t0:.1f}s")

        if val_score > best_val_metric:
            best_val_metric = val_score
            best_epoch = epoch + 1
            epochs_without_improve = 0
            best_encoder_state = copy.deepcopy(encoder.state_dict())
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= PATIENCE:
                print(f"  [{loss_name} seed={seed}] early stopping at epoch {epoch+1} "
                      f"(best epoch {best_epoch}, best val_{VAL_METRIC}={best_val_metric*100:.2f})")
                break

    # Restore the best-validation-epoch weights before final scoring.
    encoder.load_state_dict(best_encoder_state)
    user_vecs, item_vecs = get_full_embeddings(encoder, user_feat_tensor, item_feat_tensor,
                                                num_users, num_items)
    return user_vecs, item_vecs, encoder, best_epoch


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# 6. MULTI-SEED DRIVER + AGGREGATION
# ═══════════════════════════════════════════════════════════════════════
SEEDS = [42, 43, 44]
BASELINES = ["Random", "BCE", "InfoNCE", "AdvInfoNCE", "MyLoss"]


def run_all_seeds():
    data = load_coat()
    train_df = data["train_df"]
    val_ground_truth = make_ground_truth(data["val_df"])
    test_ground_truth = make_ground_truth(data["test_df"])
    r_lookup = make_r_lookup(train_df, data["num_users"], data["num_items"])
    user_seen = make_user_seen(train_df)

    n_val_users = sum(1 for v in val_ground_truth.values() if len(v) > 0)
    n_test_users = sum(1 for v in test_ground_truth.values() if len(v) > 0)
    print(f"Loaded COAT: {data['num_users']} users, {data['num_items']} items, "
          f"{len(train_df)} train interactions (train.csv, 100% intact). "
          f"random.csv split VAL_FRACTION={VAL_FRACTION} -> "
          f"{n_val_users} val users / {n_test_users} test users with "
          f">=1 positive (rating>={POS_THRESHOLD}).")

    raw = {b: defaultdict(list) for b in BASELINES}
    best_epochs = {b: [] for b in BASELINES}

    for seed in SEEDS:
        for baseline in BASELINES:
            user_vecs, item_vecs, encoder, best_epoch = train_stage1(
                baseline, seed, data, train_df, r_lookup, user_seen, val_ground_truth)

            m_test = evaluate_retriever_full(user_vecs, item_vecs, test_ground_truth,
                                              user_seen, K=HIT_K)
            for k, v in m_test.items():
                raw[baseline][k].append(v)
            best_epochs[baseline].append(best_epoch)

    return raw, best_epochs, data, train_df, val_ground_truth, test_ground_truth, user_seen


def summarize(raw):
    rows = []
    for baseline, metrics in raw.items():
        for metric, values in metrics.items():
            arr = np.array(values) * 100.0
            rows.append({"baseline": baseline, "metric": metric,
                         "mean": arr.mean(), "std": arr.std(ddof=1) if len(arr) > 1 else 0.0,
                         "n_seeds": len(arr)})
    df = pd.DataFrame(rows).sort_values(["metric", "baseline"])
    return df


def print_summary_table(df, best_epochs):
    print("\n" + "#" * 60)
    print(f"# COAT — ALL-RANKING PROTOCOL (unseen items only) — Hit@{HIT_K} / NDCG@{HIT_K}")
    print("# (test-set scores, at each seed's best-validation-epoch checkpoint)")
    print("#" * 60)
    for metric in (f"Hit@{HIT_K}", f"NDCG@{HIT_K}"):
        sub = df[df["metric"] == metric]
        print(f"\n{metric}")
        print("-" * 40)
        for _, r in sub.iterrows():
            print(f"  {r['baseline']:<12} {r['mean']:5.2f} +/- {r['std']:4.2f}  (n={r['n_seeds']})")

    print("\nMean best (early-stopped) epoch per baseline:")
    for b, eps in best_epochs.items():
        if b == "Random":
            continue
        print(f"  {b:<12} {np.mean(eps):.1f}  (seeds: {eps})")


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# 7. MAIN   (Stage-2 intentionally omitted for COAT -- no timestamps)
# ═══════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    raw, best_epochs, data, train_df, val_ground_truth, test_ground_truth, user_seen = run_all_seeds()
    summary_df = summarize(raw)
    print_summary_table(summary_df, best_epochs)
    summary_df.to_csv("coat_stage1_hit20_results_mean_std_earlystop.csv", index=False)


COAT: num_users=290  num_items=300  train_interactions=6960  random.csv total=4640 -> val=928 / test=3712  rating range=[1, 5]  (POS_THRESHOLD=4, VAL_FRACTION=0.2)
Loaded COAT: 290 users, 300 items, 6960 train interactions (train.csv, 100% intact). random.csv split VAL_FRACTION=0.2 -> 105 val users / 223 test users with >=1 positive (rating>=4).
  [BCE seed=42] epoch 1/40 loss=1.7490 val_Hit@20=7.62 time=1.0s
  [BCE seed=42] epoch 2/40 loss=1.3381 val_Hit@20=8.57 time=0.2s
  [BCE seed=42] epoch 3/40 loss=1.2147 val_Hit@20=9.52 time=0.2s
  [BCE seed=42] epoch 4/40 loss=1.0269 val_Hit@20=12.38 time=0.2s
  [BCE seed=42] epoch 5/40 loss=0.8350 val_Hit@20=9.52 time=0.2s
  [BCE seed=42] epoch 6/40 loss=0.7136 val_Hit@20=13.33 time=0.2s
  [BCE seed=42] epoch 7/40 loss=0.6289 val_Hit@20=11.43 time=0.2s
  [BCE seed=42] epoch 8/40 loss=0.5716 val_Hit@20=15.24 time=0.2s
  [BCE seed=42] epoch 9/40 loss=0.5355 val_Hit@20=9.52 time=0.2s
  [BCE seed=42] epoch 10/40 loss=0.5046 val_Hit@20=12.38 time=0